<style>
.jp-RenderedHTMLCommon h1{color:#ff9900;font-size:2.25em}.jp-RenderedHTMLCommon h2{color:#2563a8}
.jp-RenderedHTMLCommon blockquote{border-left:6px solid #ff9900;background:#fff7e8;padding:.6em 1em}
.jp-RenderedHTMLCommon table{font-size:.9em}.jp-RenderedHTMLCommon code{color:#9a3412}
</style>

# AWS Glue Exercises
## Catalogs, classifiers, crawlers, JSON, XML, and Grok

**AWS Console challenges · bronze-only staging · solutions intentionally omitted**

> Complete the exercises by building, observing, explaining, and validating AWS Glue resources—not merely by obtaining a successful crawler status.


# Scope and working rules

- Perform the work in the designated AWS account and Region.
- Upload raw inputs only below `s3://gksdatalake/bronze/`.
- Use clear resource names containing purpose and layer.
- Apply least privilege to crawler IAM roles.
- Keep unrelated formats in separate S3 prefixes.
- Capture evidence requested in each exercise.
- Do not delete shared data or roles.
- Disable schedules and clean up disposable resources after validation.
- Current AWS Console labels may differ slightly; follow the required behavior.


# Recommended evidence format

For each exercise, record:

```text
Exercise:
AWS account / Region:
Resources created:
Configuration decisions:
Observed result:
Validation performed:
Unexpected behavior:
Correction:
Evidence references:
Cleanup status:
```

Useful evidence includes resource summaries, crawler history, Catalog schemas, S3 locations, relevant log excerpts, and short explanations. Do not expose secrets, credentials, or sensitive account information.


# Exercise 1 — Explain the Glue component model

Without creating resources, draw or describe this flow:

```text
S3 data → classifier → crawler → database/table → downstream consumer
```

For every component, state:

1. what it reads;
2. what it produces;
3. where it is scoped;
4. which IAM principal or role interacts with it;
5. what it does **not** do.

**Acceptance check:** clearly distinguish the S3 objects, Catalog metadata, database namespace, crawler execution, and later ETL processing.


# Exercise 2 — Region-awareness check

1. Select the assigned Region in the AWS Console.
2. Record the Region and AWS account identifier safely.
3. Open Glue databases, tables, classifiers, and crawlers.
4. Switch temporarily to another Region without creating resources.
5. Observe which Glue resources are visible.
6. Return to the assigned Region.

**Questions**

- Why can identical resource names exist in different Regions?
- Why is a Glue resource name alone insufficient operational evidence?
- What can go wrong when S3 data and Glue resources are planned across Regions?


# Exercise 3 — MovieLens bronze layout

Verify or create this layout:

```text
s3://gksdatalake/bronze/movielens/
├── movies/movies.csv
└── ratings/ratings.csv
```

Tasks:

1. Confirm both objects exist and are nonempty.
2. Inspect the headers and representative rows.
3. Identify delimiter, quote behavior, headers, likely types, and nullable fields.
4. Explain why the two datasets should not be stored as files in one undifferentiated folder.

**Deliverable:** a proposed schema for each file before running a crawler.


# Exercise 4 — Design a MovieLens CSV classifier

Design and create a custom CSV classifier for the MovieLens files.

Decide and justify:

- classifier name and classification label;
- delimiter;
- quote symbol;
- header behavior;
- SerDe selection;
- whether one classifier is suitable for both datasets;
- how a title containing a comma should be interpreted.

**Acceptance check:** classifier configuration is recorded, and the reason for using a custom classifier instead of relying solely on built-in inference is stated.


# Exercise 5 — Database and least-privilege role

Create or identify:

- Catalog database: `gks_movielens_bronze`
- crawler IAM role scoped to the MovieLens bronze prefixes

Document the role's conceptual permissions:

- bucket listing constrained to relevant prefixes;
- object reads under `bronze/movielens/`;
- KMS decrypt when applicable;
- required Glue Catalog operations;
- Glue service trust relationship.

**Question:** which permission belongs to the console operator rather than the crawler role when assigning that role?


# Exercise 6 — Crawl MovieLens into two tables

Create a crawler that produces separate `movies` and `ratings` tables.

Requirements:

- two explicit S3 targets or another justified isolation design;
- custom CSV classifier ordered appropriately;
- output database `gks_movielens_bronze`;
- on-demand schedule;
- full initial crawl;
- bronze locations only.

After the run, validate each table's location, classification, SerDe, columns, types, and partitions.

**Acceptance check:** two correctly located tables exist and `movieId` is compatible across them.


# Exercise 7 — Find inference risks

Inspect the two MovieLens Catalog schemas and identify at least four inference risks.

Consider:

- identifiers inferred as numbers;
- rating precision;
- Unix epoch seconds versus timestamp types;
- quoted movie titles;
- `genres` as one pipe-delimited string;
- header recognition;
- future missing or malformed values.

**Deliverable:** for each risk, record the observed type, preferred business type, and where correction should occur: source contract, Catalog metadata, or a later Glue job.


# Exercise 8 — Controlled schema evolution

Use a disposable MovieLens test prefix under bronze.

1. Record the original table version/schema.
2. Add a consistent new column to the staged test data.
3. Run with normal schema updates and record the result.
4. Configure **add new columns only** and test another compatible addition.
5. Configure **log/ignore changes** and introduce one more change.
6. Compare crawler history, logs, and Catalog schema after each run.

**Acceptance check:** explain which policy fits raw discovery and which better protects a stable schema contract.


# Exercise 9 — Deletion behavior

Use disposable objects and metadata only.

1. Record a test table or partition created by a crawler.
2. Choose a crawler deletion policy deliberately.
3. Remove or relocate the corresponding disposable S3 object.
4. Recrawl and observe whether metadata is deleted, deprecated, or retained.
5. Confirm the outcome in the Catalog and crawler logs.

**Questions**

- Does deleting Catalog metadata delete the S3 object?
- Why can aggressive deletion behavior be risky?
- What should be authoritative: source layout, explicit schema code, or crawler state?


# Exercise 10 — Stage nested JSON orders

Use the JSON order document from D342 and upload it only to:

```text
s3://gksdatalake/bronze/ecommerce/orders_json/orders.json
```

Before upload:

1. validate JSON syntax and UTF-8 encoding;
2. identify wrapper-level fields;
3. identify the repeating order records;
4. predict nested structs, arrays, numeric fields, and nullable fields;
5. record the JSONPath that should select one row per order.

**Acceptance check:** the proposed path selects orders rather than the entire wrapper document.


# Exercise 11 — Classify and crawl JSON

Create a custom JSON classifier and a dataset-specific crawler.

Requirements:

- classifier name describes orders and JSON;
- record path selects each order;
- crawler targets only `bronze/ecommerce/orders_json/`;
- output database represents the bronze e-commerce domain;
- schedule remains on demand.

Validate:

- one logical row per order;
- nested customer/address structures;
- items as a repeated collection;
- nullable coupon behavior;
- whether wrapper fields appear.

**Deliverable:** predicted schema versus observed Catalog schema.


# Exercise 12 — JSONPath fault injection

Use a disposable classifier/crawler or test table.

1. Configure a path that selects the document root.
2. Crawl and observe the resulting schema.
3. Configure a path that selects the orders array elements.
4. Crawl into a separate test table.
5. Compare record boundary and nested fields.

**Questions**

- Which path produced wrapper fields?
- Which path produced one row per order?
- Why is a successful classifier match not enough to prove the correct row boundary?


# Exercise 13 — Stage XML invoices

Use the XML invoice document from D342 and upload it only to:

```text
s3://gksdatalake/bronze/ecommerce/invoices_xml/invoices.xml
```

Before upload:

1. verify one root element and balanced tags;
2. identify the repeating row element;
3. distinguish attributes from child elements;
4. identify repeated line items;
5. predict which wrapper attributes may not appear at invoice-row level.

**Deliverable:** the exact row tag and a diagram of the expected nested structure.


# Exercise 14 — Classify and crawl XML

Create a custom XML classifier and a separate XML crawler.

Requirements:

- row tag identifies one invoice;
- S3 target contains XML only;
- crawler writes to the bronze e-commerce database;
- IAM role can read only the required bronze prefix where practical.

Validate:

- invoice attributes;
- customer structure;
- repeated line items;
- tax attributes;
- inferred numeric fields;
- table location and classification.

**Acceptance check:** explain why a row tag is not an XPath expression.


# Exercise 15 — XML fault injection

Create disposable copies and test these faults separately:

- row tag with incorrect case;
- wrapper tag selected instead of repeating invoice tag;
- unescaped ampersand inside a description;
- one missing closing tag;
- an empty self-closing element used as the intended row.

For each case, predict and then record:

1. classifier/crawler outcome;
2. Catalog effect;
3. relevant log evidence;
4. minimum correction.

Do not mix malformed files into the valid shared prefix.


# Exercise 16 — Compare JSON and XML parsing

Create a comparison table covering:

| Question | JSON orders | XML invoices |
|---|---|---|
| How is a record selected? | | |
| How are nested objects represented? | | |
| How are repeated values represented? | | |
| How are attributes represented? | | |
| What wrapper data may be omitted? | | |
| Which malformed-input cases are common? | | |
| What should be normalized later? | | |

**Acceptance check:** distinguish metadata discovery from flattening and transformation.


# Exercise 17 — Inspect the Iranian access logs

Local source:

```text
C:\data\weblogs\sample-ir-logs.txt
```

Tasks:

1. inspect at least 20 varied lines;
2. identify stable delimiters and quoted sections;
3. identify the extension beyond standard combined Apache logs;
4. find examples with `-`, query strings, bots, mobile agents, and percent encoding;
5. propose column names and types before writing Grok.

**Deliverable:** one annotated line mapping every token to a proposed column.


# Exercise 18 — Build the web-log Grok classifier

Create a Grok expression for the Iranian web logs without copying it blindly.

Requirements:

- capture client IP/host, identity values, event time, HTTP method, request target, HTTP version, status, response size, referrer, user agent, and trailing forwarded field;
- preserve quoted fields containing spaces;
- handle `-` where observed;
- apply numeric casts only when valid across the source;
- keep the full pattern on one physical line.

**Acceptance check:** test representative lines and explain why `GREEDYDATA` should not appear before fields that still need parsing.


# Exercise 19 — Crawl the web logs

Upload a representative sample only to:

```text
s3://gksdatalake/bronze/logs/iranian_weblogs/
```

Create a dataset-specific crawler using the Grok classifier and write to `gks_logs_bronze`.

Validate:

- table classification and location;
- expected column count and ordering;
- numeric status/bytes where possible;
- intact request target, referrer, and user agent;
- crawler history and unmatched-line evidence;
- absence of ZIP, CSV lookup, or notebook files in the target prefix.


# Exercise 20 — Custom application-log Grok

Use the checkout events from D343.

Create reusable custom patterns for:

- order ID;
- three-letter currency;
- service name.

Create a main pattern capturing:

- ISO timestamp;
- log level;
- service;
- UUID request ID;
- order ID;
- typed amount;
- currency;
- typed latency;
- free-text message.

**Acceptance check:** the literal pipe delimiters are correctly escaped, and the message capture appears last.


# Exercise 21 — Grok variation challenge

Add these records one at a time in a disposable file:

```text
amount=-
latency_ms=-
service=checkout-v2
message=Payment authorized | gateway=primary
```

For each variation:

1. predict whether the current pattern matches;
2. run the test;
3. decide whether to broaden the Grok pattern, capture as string, or reject/quarantine the record;
4. document the impact on type safety.

**Deliverable:** revised pattern or an explicit decision not to accept the variation.


# Exercise 22 — Classifier ordering

Create a controlled test with two classifiers that could plausibly recognize the same line:

- one precise classifier;
- one deliberately broad classifier.

Attach them in one order, crawl into a disposable table, then reverse the order and crawl using a fresh crawler/table.

Observe classification and schema differences.

**Questions**

- Why should the most specific classifier generally run first?
- What does classifier certainty influence?
- Why is a new crawler safer when correcting classifier behavior?


# Exercise 23 — Troubleshooting scenarios

For each symptom, provide a diagnostic sequence rather than only a guessed cause:

1. crawler succeeds but creates one table instead of two;
2. crawler reports access denied;
3. CSV columns appear as `col0`, `col1`, and `col2`;
4. JSON produces one wrapper record;
5. XML produces no invoice rows;
6. Grok classification becomes `UNKNOWN`;
7. Catalog schema updates unexpectedly after a recrawl;
8. the correct resources appear missing in the console.

**Acceptance check:** every sequence includes configuration, source data, Region, IAM, and log/history checks where relevant.


# Exercise 24 — Crawler strategy decision

Choose a recrawl strategy for each case and justify it:

1. small static MovieLens files during development;
2. daily append-only folders partitioned by date;
3. a very large S3 dataset with event notifications configured;
4. a dataset undergoing a breaking schema redesign;
5. a curated table whose schema must not be overwritten.

Options to evaluate:

- crawl everything;
- crawl new folders only;
- event-based crawling;
- add columns only;
- log/ignore schema changes;
- explicit Catalog management instead of recurring crawls.


# Exercise 25 — Security review

Review one crawler role and resource set.

Identify:

- trust policy principal;
- S3 bucket and object scopes;
- KMS permissions;
- Glue Catalog permissions;
- permission to pass the role;
- Lake Formation involvement, if enabled;
- cross-account or cross-Region assumptions;
- permissions broader than the exercise requires.

**Deliverable:** a least-privilege improvement proposal. Do not change a shared role without authorization.


# Exercise 26 — Metadata versus data

Perform read-only inspection and answer:

- Where are the physical bytes stored?
- Where are database/table/partition definitions stored?
- What happens to S3 data if a Catalog table is deleted?
- What happens to a Catalog table if one S3 object is deleted?
- Can changing a Catalog column type rewrite historical files?
- Which component performs transformation?
- Which component discovers schema?

**Acceptance check:** no answer treats a Glue Catalog database as a running relational database engine.


# Exercise 27 — End-to-end architecture proposal

Propose a pipeline using the staged datasets:

```text
bronze raw files/logs
→ classifiers and crawlers
→ Catalog metadata
→ later Glue transformation jobs
→ validated non-bronze outputs
```

The proposal must cover:

- resource naming and Region;
- database boundaries;
- classifier ownership;
- crawler grouping and schedules;
- schema-change policy;
- rejected-record strategy;
- timestamp and monetary typing;
- monitoring and data-quality signals;
- replay/idempotency;
- cost controls.

Do not implement non-bronze writes in this exercise set.


# Exercise 28 — Cost and operational hygiene

Inspect the created resources and identify avoidable cost or operational risk:

- unnecessary crawler schedules;
- repeated full crawls of unchanged large prefixes;
- mixed datasets causing excess scanning;
- verbose logs retained indefinitely;
- duplicate classifiers/crawlers/tables;
- unused IAM roles;
- forgotten test objects;
- cross-Region data access.

**Deliverable:** a cleanup and scheduling plan that preserves required evidence while avoiding continuous crawler usage.


# Final practical assessment

Starting with a new, small line-oriented dataset:

1. place it in a new bronze-only prefix;
2. identify format and record boundary;
3. choose built-in versus CSV/JSON/XML/Grok classification;
4. create the required classifier if custom behavior is needed;
5. create a least-privilege crawler;
6. populate the correct Catalog database;
7. validate schema and location;
8. demonstrate one controlled schema change;
9. diagnose one intentionally introduced fault;
10. clean up disposable resources.

**Constraint:** the dataset and classifier choice must differ from at least one earlier exercise.


# Submission checklist

- [ ] All uploads are below `s3://gksdatalake/bronze/`
- [ ] Account and Region recorded safely
- [ ] Resource names and purposes documented
- [ ] Classifier decisions justified
- [ ] Crawler targets and ordering verified
- [ ] IAM role reviewed for least privilege
- [ ] Table locations, schemas, formats, and SerDes checked
- [ ] Schema-change and recrawl behavior demonstrated
- [ ] Failures diagnosed with evidence
- [ ] Raw data distinguished from Catalog metadata
- [ ] Disposable schedules disabled
- [ ] Cleanup status recorded
- [ ] No credentials or sensitive identifiers included in evidence


# Evaluation rubric

| Area | Weight | Evidence of completion |
|---|---:|---|
| Conceptual accuracy | 15% | components and boundaries explained correctly |
| Classifier design | 20% | CSV, JSON/XML, and Grok choices justified |
| Crawler configuration | 20% | correct targets, database, policies, and grouping |
| Schema validation | 15% | observed results compared with predictions |
| Troubleshooting | 15% | systematic diagnosis supported by logs/history |
| Security and Region | 10% | least privilege and regional scope addressed |
| Cleanup/documentation | 5% | evidence complete and disposable resources handled |

Successful resource creation alone is insufficient without validation and explanation.
